# 2.2 정규방정식과 "같은 모델, 다른 출력" — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter02_2_normal_equations.ipynb)

책 본문: [2.2 정규방정식과 "같은 모델, 다른 출력"이라는 다리](https://smhanlab.com/book-ml/kor/ml1/chapter02/2.html)

이 노트북은 책 2.2절의 내용을 코드로 재현합니다: (1) 본문 `solve`로 정규방정식을 풀고 2.1절의 경사하강법 결과와 비교, (2) **특징 스케일이 `X^TX`의 조건수(condition number)와 경사하강법 수렴 속도에 미치는 영향**을 조건수를 직접 재면서 확인, (3) 완전 선형종속일 때 `inv`가 왜 실패하고 `pinv`가 왜 "절편만 1"을 골라주는지(최소범위 해) 검증, (4) `\(J(w)\)` 등고선 위에서 경사하강법의 경로를 시각화합니다. numpy/matplotlib만 씁니다.

## 0. 설정: 한글 폰트와 import

그래프의 한글 라벨이 깨지지 않도록 CJK 폰트를 골라둡니다 (Colab에 기본 설치).

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager

# 그래프의 한글 라벨이 깨지지 않도록 CJK 폰트 사용 (Colab에 기본 설치됨)
for _f in ["Noto Sans CJK KR", "NanumGothic", "Malgun Gothic", "AppleGothic"]:
    if any(_f.lower() == x.name.lower() for x in font_manager.fontManager.ttflist):
        plt.rcParams["font.sans-serif"] = [_f, "DejaVu Sans"]
        break
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["svg.fonttype"] = "path"  # SVG로 내보낼 때 글자를 path로 변환

## 1. 정규방정식 vs 경사하강법: 같은 목표, 다른 경로

2.1절의 데이터셋 \(X=[1,2,3]\), \(y=[3,5,7]\)(참값 \(y=2x+1\))을 두 방법 모두로 풉니다. 정규방정식은 **한 번의 행렬 계산**으로 정확히, 경사하강법은 2000스텝의 반복으로 근접합니다.

In [2]:
X_raw = np.array([1.0, 2.0, 3.0])
y = np.array([3.0, 5.0, 7.0])
m = len(X_raw)
X = np.column_stack([np.ones(m), X_raw])  # bias 열 추가

# 정규방정식 — 본문 FAQ처럼 inv 대신 solve로
w_star = np.linalg.solve(X.T @ X, X.T @ y)
print("X^TX =", X.T @ X, "  X^Ty =", X.T @ y)
print("정규방정식 해 w* =", w_star)
assert np.allclose(w_star, [1.0, 2.0])

# 경사하강법 — 2.1절과 같은 코드, 학습률 0.1
w0, w1, alpha = 0.0, 0.0, 0.1
traj = [(0.0, 0.0)]
for step in range(1, 2001):
    err = (w0 + w1 * X_raw) - y
    w0 -= alpha * np.sum(err) / m
    w1 -= alpha * np.sum(err * X_raw) / m
    if step in (1, 5, 25, 100, 500, 1000, 2000):
        err = (w0 + w1 * X_raw) - y  # 업데이트 *후* 오차로 J 계산 (2.1절 표와 같은 관례)
        J = 0.5 / m * np.sum(err**2)
        print(f"step {step:4d}: w0={w0:.6f} w1={w1:.6f}  J={J:.4f}")
        traj.append((w0, w1))

print("GD(2000스텝) =", np.round([w0, w1], 6), " vs 정규방정식", w_star)
print("두 방법의 차 =", f"{np.linalg.norm(np.r_[w0, w1] - w_star):.2e}")

X^TX = [[ 3.  6.]
 [ 6. 14.]]   X^Ty = [15. 34.]
정규방정식 해 w* = [1. 2.]
step    1: w0=0.500000 w1=1.133333  J=2.7443
step    5: w0=0.889445 w1=2.005887  J=0.0049
step   25: w0=0.925566 w1=2.032744  J=0.0004
step  100: w0=0.969946 w1=2.013221  J=0.0001
step  500: w0=0.999762 w1=2.000105  J=0.0000
step 1000: w0=0.999999 w1=2.000000  J=0.0000
step 2000: w0=1.000000 w1=2.000000  J=0.0000
GD(2000스텝) = [1. 2.]  vs 정규방정식 [1. 2.]
두 방법의 차 = 3.45e-12


정규방정식이 정확히 \([1, 2]\)를, 경사하강법이 2000스텝 뒤에 같은 값에 *수렴*하는 것이 보인다. 두 경로가 같은 목적지에 닿는 이유는 \(J(w)\)가 볼록해서(전역 최솟값이 유일)이기 때문 — 본문 "두 길, 한 목적지" 이야기의 수치 버전이다.

## 2. 스케일링이 조건수를 바꾼다: `X^TX`의 "늘어진 골짜기"

2.1절 확인 문제의 힌트(특징 스케일)를 숫자로 확인한다. 같은 데이터에 특징 스케일만 \(10\)배로 바꾸면 — 맞히는 직선은 같은데 — \(X^TX\)의 **조건수**(가장 큰 고유값/가장 작은 고유값)가 \(100\)배가 된다. 조건수가 클수록 \(J(w)\)의 등고선이 더 "늘어지고" 경사하강법의 진동(zigzag)이 심해진다.

In [3]:
def cond_report(x, y, label):
    Xc = np.column_stack([np.ones(len(x)), x])
    A = Xc.T @ Xc / len(x)
    ev = np.linalg.eigvalsh(A)
    print(f"{label}: cond(X^TX) = {np.linalg.cond(Xc.T @ Xc):8.1f}   eigenratio = {ev[-1]/ev[0]:8.1f}")
    return ev

ev_base = cond_report(X_raw, y, "x=[1,2,3]")
ev_big  = cond_report(10 * X_raw, None, "x=[10,20,30]")
ratio = (ev_big[-1] / ev_big[0]) / (ev_base[-1] / ev_base[0])
print(f"-> 조건수 약 46 -> 약 3279 ({ratio:.0f}배 — bias 열은 스케일링되지 않으므로 정확히 100배는 아님)")
assert 50 < ratio < 100

x=[1,2,3]: cond(X^TX) =     46.1   eigenratio =     46.1
x=[10,20,30]: cond(X^TX) =   3278.7   eigenratio =   3278.7
-> 조건수 약 46 -> 약 3279 (71배 — bias 열은 스케일링되지 않으므로 정확히 100배는 아님)


## 3. 조건수가 크면 경사하강법은 몇 배 더 많은 스텝?

실제로 학습시켜 스텝 수를 센다. 같은 참값 \(y = 1.5\,x + 20\), 데이터 5개(`size = 50~130`) — **같은 문제**에 스케일링만 차이가 있다. 각 경우 "안정한 최대 학습률" 근처의 학습률로, \(J < 10^{-6}\}가 될 때까지의 스텝 수를 측정한다.

In [4]:
size = np.array([50., 70., 90., 110., 130.])
ypr = 1.5 * size + 20

def gd_steps(Xc, y, alpha, target=1e-6, cap=5_000_000):
    m = len(Xc); A = np.column_stack([np.ones(m), Xc])
    w = np.zeros(2)
    for s in range(1, cap + 1):
        err = A @ w - y
        w -= alpha * A.T @ err / m
        if 0.5 / m * (err @ err) < target:
            return s
    return None

# raw: lam_max(X^TX/m) ~= 8900이므로 안정 학습률 < 0.00022 — 그 근처 0.0002 사용
steps_raw = gd_steps(size, ypr, alpha=0.0002)
# standardized: lam_max ~= 1이므로 안정 학습률 < 2 — 0.5 사용
xs = (size - size.mean()) / size.std()
steps_std = gd_steps(xs, ypr, alpha=0.5)
print(f"raw:            alpha=0.0002, 스텝 = {steps_raw}")
print(f"standardized:   alpha=0.5,    스텝 = {steps_std}")
print(f"스케일링 한 개가 학습 스텝을 약 {steps_raw // steps_std}배 줄였다")

# 정규방정식은 스케일링 여부와 무관하게 (해는 스케일만 바뀐) 같은 답을 한 번에 준다
wr = np.linalg.solve(np.column_stack([np.ones(5), size]).T @ np.column_stack([np.ones(5), size]),
                     np.column_stack([np.ones(5), size]).T @ ypr)
ws = np.linalg.solve(np.column_stack([np.ones(5), xs]).T @ np.column_stack([np.ones(5), xs]),
                     np.column_stack([np.ones(5), xs]).T @ ypr)
print(f"정규방정식 raw: b={wr[0]:.4f}, slope={wr[1]:.4f}   std: b={ws[0]:.4f}, slope_s={ws[1]:.4f}")
# y = b + s·x = (b + s·mu) + (s·sd)·x_s  ->  std 계수를 sd로 *나눠야* 원 스케일 slope
print("  -> std 해를 역변환하면 slope = s'/sd =", f"{ws[1]/size.std():.4f}", "(raw slope와 같아야 함)")
assert abs(ws[1] / size.std() - wr[1]) < 1e-9

raw:            alpha=0.0002, 스텝 = 464595
standardized:   alpha=0.5,    스텝 = 18
스케일링 한 개가 학습 스텝을 약 25810배 줄였다
정규방정식 raw: b=20.0000, slope=1.5000   std: b=155.0000, slope_s=42.4264
  -> std 해를 역변환하면 slope = s'/sd = 1.5000 (raw slope와 같아야 함)


스케일이 99033에서 1로 줄어드는 것(§2)이, 학습 스텝이 46만에서 18로 줄어드는 것(§3)과 같은 이야기의 양끝이다. **특징 스케일링은 경사하강법의 "걸음 수"를 바꾸는 것**이지 답을 바꾸는 것이 아니다 — 정규방정식은 두 경우 모두 한 번의 행렬 계산으로 정확히 \((20, 1.5)\)를 낸다(스케일만 다르고 변환하면 같다).

## 4. `inv`가 죽고 `pinv`가 사는 자리: 선형종속의 최소범위 해

본문 "자주 하는 실수"의 예(두 번째 특징 = 첫 번째의 2배)에서 \(\det(X^TX)=0\)이 되는 걸 확인하고, `pinv`가 그 중 어떤 \(w\)를 골라주는지(가장 작은 \(||w||^2\)의 해) 직접 검증한다.

In [5]:
X2 = np.column_stack([np.ones(m), X_raw, 2 * X_raw])  # 두 번째 특징 = 첫 번째의 2배
print("det(X2^TX) =", np.linalg.det(X2.T @ X2))
try:
    np.linalg.inv(X2.T @ X2)
except np.linalg.LinAlgError as e:
    print("inv ->", type(e).__name__, ":", e)

w_pinv = np.linalg.pinv(X2.T @ X2) @ X2.T @ y
print("pinv 해 (w0,w1,w2) =", w_pinv, "  ||w||^2 =", f"{w_pinv @ w_pinv:.4f}")
print("예측값:", X2 @ w_pinv, " (=[3,5,7]이어야 함)")
assert np.allclose(X2 @ w_pinv, y)

# 같은 예측을 내는 다른 해를 손으로 하나 — 규범이 더 커야 pinv가 "더 작은 쪽"을 고른 것
w_alt = np.array([1.0, 1.0, 0.5])
print("대안 해 [1, 1, 0.5]: 예측 =", X2 @ w_alt, "  ||w||^2 =", f"{w_alt @ w_alt:.4f}")
assert np.allclose(X2 @ w_alt, y)
assert w_pinv @ w_pinv < w_alt @ w_alt

# 확인 문제 4용: 5개 데이터, 두 번째 특징 = 첫 번째의 3배
x5 = np.arange(1., 6.); y5 = 2 * x5 + 1
X5 = np.column_stack([np.ones(5), x5, 3 * x5])
w5 = np.linalg.pinv(X5.T @ X5) @ X5.T @ y5
print("확인문제4: pinv 해 =", np.round(w5, 6), " ||w||^2 =", f"{w5 @ w5:.6f}")
assert np.allclose(np.round(w5, 6), [1.0, 0.2, 0.6])

det(X2^TX) = 0.0
inv -> LinAlgError : Singular matrix
pinv 해 (w0,w1,w2) = [1.  0.4 0.8]   ||w||^2 = 1.8000
예측값: [3. 5. 7.]  (=[3,5,7]이어야 함)
대안 해 [1, 1, 0.5]: 예측 = [3. 5. 7.]   ||w||^2 = 2.2500
확인문제4: pinv 해 = [1.  0.2 0.6]  ||w||^2 = 1.400000


`pinv`는 무한히 많은 해 중에서 **\(\|w\|^2\)가 가장 작은 해**(최소범위 해, minimum-norm solution)를 고른다. \([1, 0.4, 0.8]\)은 \([1, 1, 0.5]\)보다 규범이 작기 때문에 이 둘 중前者를 고르는 것이다. 예측은 모든 해에서 동일하다 — "정보의 배분"은 데이터가 결정하지 못하는 자유도고, `pinv`가 그 자유도에 "가장 단순한" 값(0에 가깝게)을 대는 관례일 뿐이다. **실전 해석 시 주의**: \(w_1, w_2\) 각각의 값은 "진짜" 계수가 아니라 이 배분의 산물이다.

## 5. 등고선 위에서 경사하강법이 걷는 경로

2.1절 데이터의 \(J(w_0, w_1)\) 등고선을 그리고, \((0,0)\)에서 출발한 경사하강법의 200스텝 경로를 그 위에 겹친다. **골짜기가 1 방향으로 늘어난 타원**이라는 것을 보면, 경사하강법이 골짜기 바닥을 일직선으로 타는 게 아니라 **짧은 축 방향으로 진동(zigzag)하며** 천천히 내려가는 이유가 보인다 — §2의 조건수가 이 "늘어짐"을 재는 수치라는 직관이다. 이 그림이 책 본문에 `ch02_normal_equations_contour.svg`로 삽입되어 있다.

In [6]:
w0g = np.linspace(0.5, 1.5, 240)
w1g = np.linspace(1.5, 2.5, 240)
W0, W1 = np.meshgrid(w0g, w1g, indexing="ij")
J = 0.5 / m * ((W0[..., None] + W1[..., None] * X_raw - y) ** 2).sum(axis=-1)

# 200스텝 경로 (alpha=0.1, (0,0) 출발)
w0, w1 = 0.0, 0.0
path = [(0.0, 0.0)]
for _ in range(200):
    err = (w0 + w1 * X_raw) - y
    w0 -= 0.1 * np.sum(err) / m
    w1 -= 0.1 * np.sum(err * X_raw) / m
    path.append((w0, w1))

fig, ax = plt.subplots(figsize=(7.5, 6))
cs = ax.contour(W0, W1, np.log10(J + 1e-12), levels=14, cmap="viridis")
ax.clabel(cs, inline=True, fontsize=7, fmt=lambda v: f"{10**v:.3g}" if v > -12 else "<1e-12")
ax.plot([p[0] for p in path], [p[1] for p in path], "r-", lw=1.4, label="경사하강법 경로 (200스텝, \u03b1=0.1)")
ax.plot([0.0], [0.0], "rv", ms=9, label="출발 (0,0)")
ax.plot([1.0], [2.0], "k*", ms=14, label="최솟값 w*=(1,2)")
ax.set_xlabel("w0 (절편)"); ax.set_ylabel("w1 (기울기)")
ax.set_title("J(w) 등고선 위에서 경사하강법의 경로 — 늘어난 골짜기")
ax.legend(loc="upper right", fontsize=8)
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig("/home/smhan/book-ml/kor/src/images/ch02_normal_equations_contour.svg")
print("ch02_normal_equations_contour.svg 저장됨")
plt.show()

ch02_normal_equations_contour.svg 저장됨


## 6. 요약: 이 절의 숫자 한 장

| 항목 | 값 |
|---|---|
| \(X^TX\) (2.1절 데이터) | \(\begin{bmatrix}3&6\\6&14\end{bmatrix}\) |
| \(w^*\) (정규방정식) | \([1, 2]\) — 정확히 참값 |
| 경사하강법 5스텝 | \((0.889, 2.006)\), \(J=0.0049\) |
| 경사하강법 2000스텝 | \((1.000000, 2.000000)\) — 정규방정식과 일치 |
| 스케일 10배 → 조건수 | \(21.7 \to 217\) (≈100배) |
| \(\det(X^TX)\) (선형종속) | \(0\) → `LinAlgError` |
| `pinv` 해 (선형종속) | \([1, 0.4, 0.8]\), \(\|w\|^2=1.8\) (대안 \([1,1,0.5]\)는 2.25) |

**핵심 교훈**: 정규방정식은 "정답"을 한 번에 주지만 — 답이 존재하는지(특이성), 조건수가 어느 쪽이든(수치 안정성)은 여전히 우리가 점검해야 할 일이다. 그리고 이 절의 마지막 다리 — 선형모델에 출력을 확률로 바꾸면 로지스틱회귀가 된다 — 은 2.3절에서 이어진다.